In [ ]:
# 1. Instalación de todo lo necesario (una sola vez)
!pip install -q transformers datasets evaluate accelerate peft bitsandbytes sentencepiece rouge_score

# reiniciar entorno
import os
os.kill(os.getpid(), 9)

In [ ]:
# 2. Importar todo
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate
import numpy as np
import gc
torch.cuda.empty_cache()
gc.collect()
print("Todo listo")

Todo listo


In [ ]:
# 3. Summarization – Caso de uso: textos científicos (arXiv)
print("CARGANDO MODELO 1: Summarization científica")

model_name_summ = "facebook/bart-large-cnn"
tokenizer_summ = AutoTokenizer.from_pretrained(model_name_summ)
model_summ = AutoModelForSeq2SeqLM.from_pretrained(model_name_summ, torch_dtype=torch.float16)

# Aplicar LoRA
model_summ = prepare_model_for_kbit_training(model_summ)
lora_config_summ = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none", task_type="SEQ_2_SEQ_LM"
)
model_summ = get_peft_model(model_summ, lora_config_summ)
print("Parámetros entrenables:", model_summ.print_trainable_parameters())

CARGANDO MODELO 1: Summarization científica


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

trainable params: 2,359,296 || all params: 408,649,728 || trainable%: 0.5773
Parámetros entrenables: None


In [ ]:
# 4. Cargar dataset arXiv y entrenar
dataset_summ = load_dataset("ccdv/arxiv-summarization", split="train[:800]")

def preprocess_summ(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]
    model_inputs = tokenizer_summ(inputs, max_length=1024, truncation=True, padding="max_length")
    labels = tokenizer_summ(examples["abstract"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_summ = dataset_summ.map(preprocess_summ, batched=True, remove_columns=dataset_summ.column_names)

training_args = TrainingArguments(
    output_dir="./mi_bart_arxiv_lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_steps=400,
    logging_steps=50,
    report_to=[],
    push_to_hub=False
)

trainer_summ = Trainer(model_summ, training_args, train_dataset=tokenized_summ)
trainer_summ.train()

# Guardar
trainer_summ.save_model("./mi_bart_arxiv_lora")
tokenizer_summ.save_pretrained("./mi_bart_arxiv_lora")
print("MODELO 1 GUARDADO: ./mi_bart_arxiv_lora")

README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00003-of-00015.parquet:   0%|          | 0.00/227M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Step,Training Loss
50,3.545400
100,3.183200
150,2.937600
200,2.790500
250,2.770500
300,2.725300


MODELO 1 GUARDADO: ./mi_bart_arxiv_lora


In [ ]:
# MODELO 2: Clasificación detección de SMS spam
print("ENTRENANDO MODELO DE CLASIFICACIÓN SPAM")

# 1. Cargar tokenizer y modelo base
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    torch_dtype=torch.float16
)

# 2. LoRA con los target_modules para DistilBERT
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
print("Parámetros entrenables:")
model.print_trainable_parameters()

ENTRENANDO MODELO DE CLASIFICACIÓN SPAM


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parámetros entrenables:
trainable params: 887,042 || all params: 67,842,052 || trainable%: 1.3075


In [ ]:
# Cargar dataset SMS spam
dataset = load_dataset("sms_spam")

# Función de tokenización
def tokenize_function(examples):
    return tokenizer(examples["sms"], truncation=True, padding="max_length", max_length=128)

# Aplicar tokenización
tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Entrenamiento
training_args = TrainingArguments(
    output_dir="./mi_modelo_spam_final",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

print("Entrenando modelo spam...")
trainer.train()

# GUARDAR MODELO FINAL
trainer.save_model("./mi_modelo_spam_final")
tokenizer.save_pretrained("./mi_modelo_spam_final")
print("MODELO SPAM GUARDADO EN ./mi_modelo_spam_final")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

Map:   0%|          | 0/5574 [00:00<?, ? examples/s]

Entrenando modelo spam...


Step,Training Loss
50,0.164200
100,0.026200
150,0.030300
200,0.071500
250,0.033900
300,0.028600
350,0.059600
400,0.034300
450,0.028500
500,0.022200


MODELO SPAM GUARDADO EN ./mi_modelo_spam_final


In [ ]:
# 7. Evaluación oficial
rouge = evaluate.load("rouge")

# Summarization
summarizer_yours = pipeline("summarization", model="./mi_bart_arxiv_lora", device=0)
summarizer_orig = pipeline("summarization", model="facebook/bart-large-cnn", device=0)

test_summ = load_dataset("ccdv/arxiv-summarization", split="test[:5]")

preds_your = [summarizer_yours(x["article"][:1000])[0]["summary_text"] for x in test_summ]
rouge_your = rouge.compute(predictions=preds_your, references=test_summ["abstract"])["rougeL"]
print(f"TU MODELO SUMMARIZATION → ROUGE-L: {rouge_your:.3f}")

# Clasificación
classifier = pipeline("text-classification", model="./mi_modelo_spam_final", device=0)
test_spam = load_dataset("sms_spam", split="train").train_test_split(test_size=0.2)["test"]

correct = 0
for ex in test_spam:
    pred = classifier(ex["sms"])[0]
    if ("1" in pred["label"]) == (ex["label"] == 1):
        correct += 1
acc = correct / len(test_spam)
print(f"TU MODELO CLASIFICACIÓN → Accuracy: {acc*100:.2f}%")

Device set to use cuda:0
Device set to use cuda:0


TU MODELO SUMMARIZATION → ROUGE-L: 0.207


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


TU MODELO CLASIFICACIÓN → Accuracy: 99.55%


In [ ]:
from transformers import pipeline
import evaluate
from datasets import load_dataset

# sumarizacion y clasificacion
summarizer = pipeline(
    "summarization",
    model="./mi_bart_arxiv_lora",
    device=0,
    torch_dtype=torch.float16
)

classifier = pipeline("text-classification", model="./mi_modelo_spam_final", device=0)

rouge = evaluate.load("rouge")
test = load_dataset("ccdv/arxiv-summarization", split="test[:3]")
preds = [summarizer(x["article"][:1000], max_length=100)[0]["summary_text"] for x in test]
print(f"ROUGE-L Summarization: {rouge.compute(predictions=preds, references=test['abstract'])['rougeL']:.3f}")

data = load_dataset("sms_spam", split="train")
test_spam = data.train_test_split(test_size=0.2, seed=42)["test"]
correct = sum(1 for x in test_spam if ("1" in classifier(x["sms"])[0]["label"]) == (x["label"]==1))
print(f"Accuracy Spam: {correct/len(test_spam)*100:.2f}%")

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


ROUGE-L Summarization: 0.173
Accuracy Spam: 99.55%


# **Pruebas**

In [ ]:
# 1. SUMMARIZATION – Scientific papers
summarizer = pipeline(
    "summarization",
    model="./mi_bart_arxiv_lora",
    device=0,
    torch_dtype=torch.float16
)
print("Scientific summarization model loaded")

Device set to use cuda:0


Scientific summarization model loaded


In [ ]:
print("="*90)
print("LIVE DEMO – Prof. Carlos B. Ogando Project")
print("="*90)

# 1. Scientific Summarization
text = "Researchers have developed a revolutionary quantum algorithm that dramatically accelerates molecular dynamics simulations. Using tensor network methods on a 100-qubit quantum processor, the team achieved exponential speedup compared to classical supercomputers."
print("SCIENTIFIC SUMMARIZATION:")
print(summarizer(text, max_length=80, min_length=30, do_sample=False)[0]["summary_text"])

# 2. SMS Spam Classification
classifier = pipeline("text-classification", model="./mi_modelo_spam_final", device=0)

examples = [
    "Congratulations! You've won a free iPhone 16 PRO MAX!",
    "Hey, are we still meeting tomorrow at 8pm?",
    "URGENT: Your account will be suspended. Click here to verify",
    "I'll pick you up at 6 for class"
]

print("\nSMS SPAM DETECTION:")
for sms in examples:
    result = classifier(sms)[0]
    label = "SPAM" if "1" in result["label"] else "NOT SPAM"
    print(f"{label:8} → {sms} (confidence: {result['score']:.1%})")

Your max_length is set to 80, but your input_length is only 44. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=22)


LIVE DEMO – Prof. Carlos B. Ogando Project
SCIENTIFIC SUMMARIZATION:
a revolutionary quantum algorithm that dramatically accelerates molecular dynamics simulations. Using tensor network methods on a 100-qubit quantum processor, the team achieved exponential speedup compared to classical supercomputers.


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0



SMS SPAM DETECTION:
SPAM     → Congratulations! You've won a free iPhone 16 PRO MAX! (confidence: 99.1%)
NOT SPAM → Hey, are we still meeting tomorrow at 8pm? (confidence: 99.3%)
SPAM     → URGENT: Your account will be suspended. Click here to verify (confidence: 65.6%)
NOT SPAM → I'll pick you up at 6 for class (confidence: 100.0%)
